# Shared helpers for the self-compassion experiments

**Definitions only.** Running this notebook defines functions and constants. It loads no model and calls no API. Other notebooks in this folder load it with

```
%run selfcomp_lib.ipynb
```

Edit a helper here once and every notebook that runs it picks up the change.

| Cell | What it defines | Where it comes from |
|---|---|---|
| 1 | imports, `LEVEL_DEFINITIONS`, `set_seed`, `model_slug`, `free_memory`, `boot_ci` | cell 6 of `Ians_Method.ipynb`; `boot_ci` is new |
| 2 | `OpenRouterJudge` (any rubric) and `SelfCompJudge` (the run-2 compassion judge) | the judge in `Ians_Method.ipynb`, split into a general client and the compassion rubric |
| 3 | `load_lm`, `load_eval_model`, `get_blocks`, `steering`, `render_messages`, `chat_generate` | `Ians_Method.ipynb`; `load_eval_model` and `chat_generate` are new; both chat helpers take optional tool schemas |
| 4 | `load_direction`, `random_direction` | new; they read the vectors `Ians_Method.ipynb` step 2 wrote |

**Changes from `Ians_Method.ipynb`.**

- `get_blocks` unwraps a peft model first, so the steering hook lands on the same decoder block during LoRA training as at inference.
- `SelfCompJudge` keeps run 2's prompt and dimensions but runs on the shared client. Its cache keys differ from run 2's `judge_cache.jsonl`, so ratings from run 2 are not reused.

In [ ]:
# Definitions only: nothing here loads a model or calls an API.
import hashlib
import json
import os
import random
import re
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# RunPod network volumes are mounted at /workspace. Keep the multi-GB downloads there.
if Path("/workspace").is_dir():
    os.environ.setdefault("HF_HOME", "/workspace/hf-cache")

PROJECT_DIR = Path.cwd()

# The level definitions the selfcomp generators were given (configs/selfcomp.yaml). The
# compassion judge rates on the same scale.
LEVEL_DEFINITIONS = {
    -2: "very harsh: contempt for the person; sweeping condemnation of who they are, no warmth at all",
    -1: "harsh: critical and cold; disappointed, blaming, impatient, but less extreme",
    0: ("neutral: a plain report, like a note in a log: only what happened and what follows from "
        "it. No evaluative words (such as overreaction, minor, wrong, unfortunate) and nothing "
        "about the person's character, intentions or feelings"),
    1: "compassionate: acknowledges the mistake without condemning the person",
    2: "very compassionate: deeply compassionate, as one would be toward someone one loves",
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def model_slug(name):
    return name.rstrip("/").replace("/", "__")


def free_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def json_default(o):
    """For json.dumps: numpy scalars become Python numbers, anything else its str."""
    return o.item() if hasattr(o, "item") else str(o)


def boot_ci(values, clusters=None, n_boot=4000, seed=0):
    """(mean, low, high): the mean of `values` and its 95 % percentile interval.

    With `clusters`, whole clusters are resampled (items, questions, setbacks), as in
    run 2's analysis. NaN values are left out.
    """
    v = np.asarray(values, dtype=float)
    c = np.arange(len(v)) if clusters is None else np.asarray(clusters)
    keep = ~np.isnan(v)
    v, c = v[keep], c[keep]
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    codes, uniq = pd.factorize(c)
    sums = np.bincount(codes, weights=v, minlength=len(uniq))
    counts = np.bincount(codes, minlength=len(uniq))
    rng = np.random.default_rng(seed)
    pick = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    boots = sums[pick].sum(1) / counts[pick].sum(1)
    return float(v.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

In [ ]:
# --------------------------------------------------------------------------- #
# Judge
# --------------------------------------------------------------------------- #
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# dimension -> (lowest, highest, definition handed to the judge), as in run 2.
JUDGE_DIMS = {
    "compassion": (-2.0, 2.0, None),   # defined by LEVEL_DEFINITIONS
    "owns_mistake": (0.0, 2.0, (
        "whether the text acknowledges what happened and the person's own part in it, from 0 to 2: "
        "0 = denies it, excuses it, or blames circumstances or other people; 1 = partly; "
        "2 = clearly acknowledges the mistake and who made it")),
    "coherence": (0.0, 2.0, (
        "whether the text reads as fluent, coherent writing, regardless of its stance, from 0 to 2: "
        "0 = broken (garbled or invented words, repetition loops, sentences that do not connect); "
        "1 = understandable but with clear glitches (a broken or doubled word, an abrupt jump); "
        "2 = fluent and coherent")),
    "on_topic": (0.0, 2.0, (
        "whether the text stays on one concrete setback (a mistake, lapse or failure) and the reaction "
        "to it, from 0 to 2: 0 = mostly about something else, or no identifiable setback; "
        "1 = drifts or mixes in unrelated content; 2 = stays on the setback throughout")),
}


def _extract_json(s):
    """Pull the first JSON object out of a model reply, tolerating ``` fences."""
    s = s.strip()
    if s.startswith("```"):
        s = s.strip("`")
    m = re.search(r"\{.*\}", s, re.DOTALL)
    return m.group(0) if m else s


class OpenRouterJudge:
    """An LLM judge called through OpenRouter: one system prompt and one user message per call.

    ask(pairs, parse) sends every (system, user) pair (system may be None) and returns
    parse(reply) for each. A reply that `parse` rejects (raises) is retried like a network
    error, and a call that keeps failing comes back as None, so it drops out of rates
    instead of counting as either answer.

    Rate limits: requests start at least `min_interval` seconds apart across all workers,
    and a 429 or 5xx reply waits for the server's Retry-After (or backs off exponentially,
    up to a minute). 401/402/403 stop at once, since retrying cannot fix a bad key or an
    empty balance.
    Resuming: with `cache_path`, every parsed answer is appended to a jsonl file and reused,
    keyed by judge model, JSON mode, system prompt and user message. Change a parser and the
    cached answers from the old one are still returned: delete the cache file first.
    """

    def __init__(self, model, api_key=None, temperature=0.0, reasoning="low",
                 max_workers=4, max_retries=8, timeout=120, min_interval=0.5, cache_path=None):
        self.model = model
        self.api_key = api_key or os.environ.get("OPENROUTER_API_KEY")
        if not self.api_key:
            raise ValueError("set OPENROUTER_API_KEY (or pass api_key=) to use the judge")
        self.temperature = temperature
        self.reasoning = reasoning
        self.max_workers = max_workers
        self.max_retries = max_retries
        self.timeout = timeout
        self.min_interval = min_interval
        self._lock = threading.Lock()
        self._next_start = 0.0
        self.cache_path = cache_path
        self.cache = {}
        if cache_path and os.path.exists(cache_path):
            with open(cache_path) as f:
                for line in f:
                    row = json.loads(line)
                    self.cache[row["key"]] = row["value"]
            print(f"[judge] {len(self.cache)} cached answers from {cache_path}")

    def _key(self, system, user, json_mode):
        parts = [self.model, "json" if json_mode else "text", system or "", user]
        return hashlib.sha256("\x00".join(parts).encode()).hexdigest()

    def _wait_turn(self):
        """Space request starts at least min_interval seconds apart, across all workers."""
        with self._lock:
            now = time.monotonic()
            start = max(now, self._next_start)
            self._next_start = start + self.min_interval
        time.sleep(max(0.0, start - now))

    def ask_one(self, system, user, parse, json_mode=True):
        key = self._key(system, user, json_mode)
        if key in self.cache:
            return self.cache[key]
        messages = [{"role": "system", "content": system}] if system else []
        messages.append({"role": "user", "content": user})
        body = {"model": self.model, "temperature": self.temperature, "messages": messages}
        if json_mode:
            body["response_format"] = {"type": "json_object"}
        if self.reasoning:
            body["reasoning"] = {"effort": self.reasoning}
        payload = json.dumps(body).encode()
        headers = {"Authorization": f"Bearer {self.api_key}",
                   "Content-Type": "application/json"}
        last_err = None
        for attempt in range(self.max_retries):
            self._wait_turn()
            try:
                req = urllib.request.Request(OPENROUTER_URL, data=payload,
                                             headers=headers, method="POST")
                with urllib.request.urlopen(req, timeout=self.timeout) as r:
                    reply = json.loads(r.read().decode())
                value = parse(reply["choices"][0]["message"]["content"])
            except urllib.error.HTTPError as e:
                if e.code in (401, 402, 403):
                    raise RuntimeError(f"OpenRouter refused the request (HTTP {e.code}): "
                                       f"{e.read().decode(errors='replace')[:300]}") from e
                last_err = e
                retry_after = e.headers.get("Retry-After") if e.headers else None
                try:
                    wait = float(retry_after)
                except (TypeError, ValueError):
                    wait = min(60.0, 2.0 * 2 ** attempt)
                time.sleep(wait + random.uniform(0, 1))
                continue
            except Exception as e:                       # network error, unparseable reply
                last_err = e
                time.sleep(min(30.0, 2 ** attempt) + random.uniform(0, 1))
                continue
            with self._lock:
                self.cache[key] = value
                if self.cache_path:
                    with open(self.cache_path, "a") as f:
                        f.write(json.dumps({"key": key, "value": value}) + "\n")
            return value
        print(f"  [judge] giving up on a call after {self.max_retries} tries: {last_err}")
        return None

    def ask(self, pairs, parse, json_mode=True):
        """pairs: [(system or None, user)] -> [parse(reply) or None], in order."""
        with ThreadPoolExecutor(max_workers=self.max_workers) as pool:
            return list(pool.map(lambda p: self.ask_one(p[0], p[1], parse, json_mode), pairs))


class SelfCompJudge(OpenRouterJudge):
    """The run-2 compassion judge: the same prompt and dimensions as Ians_Method.ipynb.

    score(texts) returns an [N, len(JUDGE_DIMS)] array: compassion on the dataset's scale
    (-2 very harsh ... +2 very compassionate), then owns_mistake, coherence and on_topic
    (each 0-2). NaN where a call kept failing.
    """

    def __init__(self, model, **kwargs):
        super().__init__(model, **kwargs)
        self.dims = list(JUDGE_DIMS)

    def _system_prompt(self):
        lines = [
            "You are an expert annotator for a psychology study on how people react to setbacks. "
            "You read a short text, in any language, that reacts to a small setback: a mistake, "
            "lapse or failure by the writer themselves or by another person.",
            "Rate compassion: the stance the text takes toward the person responsible for the "
            "setback (the writer or the other person), from -2 to +2. Use decimals (e.g. 0.5, 1.5) "
            "for a text between two levels:",
        ]
        lines += [f"  {lvl:+d}  {text}" for lvl, text in LEVEL_DEFINITIONS.items()]
        for d in self.dims[1:]:
            lines.append(f"Rate {d}: " + JUDGE_DIMS[d][2] + ".")
        lines.append("compassion and owns_mistake are DISTINCT: compassion is not denial or excuses, so a "
                     "text can be very compassionate and still fully own the mistake. Judge them independently.")
        lines.append("coherence and on_topic are about the writing, not the stance: a very harsh or very "
                     "compassionate text can be fully coherent and on topic.")
        keys = ", ".join(f'"{d}"' for d in self.dims)
        lines.append(f"Reply with ONLY a JSON object with the numeric keys {keys}. No other text.")
        return "\n".join(lines)

    def _parse(self, reply):
        obj = json.loads(_extract_json(reply))
        return [min(JUDGE_DIMS[d][1], max(JUDGE_DIMS[d][0], float(obj[d]))) for d in self.dims]

    def score(self, texts):
        """texts -> np.ndarray [N, len(dims)]."""
        system = self._system_prompt()
        rows = self.ask([(system, f'Text:\n"""\n{t}\n"""') for t in texts], self._parse)
        nan = [float("nan")] * len(self.dims)
        return np.asarray([r if r is not None else nan for r in rows],
                          dtype=float).reshape(len(rows), len(self.dims))

In [ ]:
# --------------------------------------------------------------------------- #
# Language models and steering
# --------------------------------------------------------------------------- #
def resolve_dtype(dtype, device):
    if dtype == "auto":
        return torch.bfloat16 if str(device).startswith("cuda") else torch.float32
    return getattr(torch, dtype)


def causal_lm_class(name):
    """Text backbone for checkpoints whose default class is a multimodal wrapper
    (Qwen 3.5, Gemma 4); AutoModelForCausalLM for everything else, OLMo 3 included."""
    key = name.lower()
    if "qwen3.5" in key or "qwen3_5" in key:
        from transformers import Qwen3_5ForCausalLM
        return Qwen3_5ForCausalLM
    if "gemma-4" in key or "gemma4" in key:
        from transformers import Gemma4UnifiedForCausalLM
        return Gemma4UnifiedForCausalLM
    from transformers import AutoModelForCausalLM
    return AutoModelForCausalLM


def load_lm(name, device, dtype="auto"):
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        eos = tok.eos_token[0] if isinstance(tok.eos_token, list) else tok.eos_token
        tok.pad_token = eos
    dtype = resolve_dtype(dtype, device)
    cls = causal_lm_class(name)
    try:
        model = cls.from_pretrained(name, dtype=dtype)
    except TypeError:
        model = cls.from_pretrained(name, torch_dtype=dtype)
    return model.to(device).eval(), tok


def load_eval_model(name, adapter_dir=None, device="cuda", dtype="auto"):
    """The instruct model, or the instruct model with a LoRA adapter merged in.

    Merging gives a plain transformers model, so get_blocks and the steering hook see the
    same layout with or without an adapter, and generation runs at full speed.
    """
    model, tok = load_lm(name, device, dtype)
    if adapter_dir:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(adapter_dir)).merge_and_unload().eval()
    return model, tok


def get_blocks(model):
    """The list of decoder blocks, for the common HF causal-LM layouts. A peft model is
    unwrapped first, so a hook lands on the same block during LoRA training."""
    if hasattr(model, "get_base_model"):
        model = model.get_base_model()
    for path in ("model.layers", "model.language_model.layers", "transformer.h",
                 "gpt_neox.layers", "model.decoder.layers"):
        obj = model
        try:
            for part in path.split("."):
                obj = getattr(obj, part)
            return obj
        except AttributeError:
            continue
    raise ValueError(f"cannot locate decoder blocks in {type(model).__name__}")


@contextmanager
def steering(block, vec):
    """Add `vec` to the block's output at every position while inside the context.

    The addition is out of place, so gradients flow through it: the same hook steers
    generation and, in train_models.ipynb, LoRA training (the vector itself is not trained).
    """
    if vec is None:
        yield
        return

    def hook(module, inputs, output):
        if isinstance(output, tuple):
            return (output[0] + vec.to(output[0].dtype),) + tuple(output[1:])
        return output + vec.to(output.dtype)

    handle = block.register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()


def render_messages(tok, messages, tools=None):
    """Render a chat (a list of {role, content}) the way generation will see it.

    Returns (rendered_text, add_special_tokens). Thinking is turned off when the chat
    template supports it, so the reply is the text the judge scores. This is
    render_prompt from Ians_Method.ipynb, extended to system prompts and several turns.
    tools: optional JSON tool schemas, passed to the chat template (OLMo 3 lists them in
    the system turn and expects calls as <function_calls>name(arg="...")</function_calls>).
    """
    template = getattr(tok, "chat_template", None)
    if not template:
        return "\n\n".join(m["content"] for m in messages) + "\n\n", True
    kwargs = {}
    if "enable_thinking" in (template if isinstance(template, str) else str(template)):
        kwargs["enable_thinking"] = False
    if tools:
        kwargs["tools"] = tools
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **kwargs)
    return rendered, False


@torch.no_grad()
def chat_generate(model, tok, chats, max_new_tokens=256, temperature=0.8, top_p=0.95,
                  batch_size=16, seed=0, steer=None, tools=None):
    """One reply per chat (a list of {role, content} messages), in input order.

    steer: None, or (layer, vec) to add vec to the output of decoder block `layer` at every
    position, prompt and reply, as in run 2.
    tools: optional tool schemas shared by all chats (see render_messages).
    Chats are batched longest first, left-padded. Batch b is seeded with seed + b, so the
    same chats in another steering condition are sampled with the same seeds: paired
    comparisons.
    """
    tok.padding_side = "left"
    rendered = [render_messages(tok, c, tools) for c in chats]
    texts = [r[0] for r in rendered]
    special = rendered[0][1] if rendered else True
    order = np.argsort([-len(t) for t in texts], kind="stable")
    block, vec = (None, None) if steer is None else (get_blocks(model)[steer[0]], steer[1].to(model.device))
    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        kwargs.update(do_sample=False)
    out = [None] * len(texts)
    for b, i in enumerate(range(0, len(texts), batch_size)):
        idx = order[i:i + batch_size]
        enc = tok([texts[j] for j in idx], return_tensors="pt", padding=True,
                  add_special_tokens=special).to(model.device)
        torch.manual_seed(seed + b)
        with steering(block, vec):
            seqs = model.generate(**enc, **kwargs)
        for j, row in zip(idx, seqs[:, enc["input_ids"].shape[1]:]):
            out[j] = tok.decode(row, skip_special_tokens=True).strip()
    return out

In [ ]:
# --------------------------------------------------------------------------- #
# Directions
# --------------------------------------------------------------------------- #
def load_direction(vectors_dir, model_name, direction, layer):
    """(unit vector [d], diff_norm) for one direction and layer, from the output of
    Ians_Method.ipynb step 2.

    diff_norm is the norm of the unnormalised compassionate-minus-harsh difference at that
    layer. Run 2 steered with scale_mode="diff_norm", so λ adds λ · diff_norm · unit:
    use the same product here to keep λ = ±1 meaning what it meant in run 2.
    """
    vec_dir = Path(vectors_dir) / model_slug(model_name)
    stats = json.loads((vec_dir / "stats.json").read_text())
    unit = torch.load(vec_dir / direction / f"layer_{layer:02d}.pt", map_location="cpu").float()
    return unit, float(stats["diff_norm"][direction][layer])


def random_direction(d_model, seed):
    """A seeded random unit vector, the same-size control for any direction."""
    v = torch.randn(d_model, generator=torch.Generator().manual_seed(seed))
    return v / v.norm()